# 02 – Train pilot theo shard
**Cài đặt notebook:** Accelerator = **GPU T4 x2**; **Internet = On**; *Add Input* → output của notebook `01` (file `cremad_processed.tar`).

**Thứ tự chạy:**
1. **Đo thời gian (session tương tác):** đặt `TIMING_ONLY = True`, chạy từng cell. Cell cuối in ra số giờ ước tính cho mỗi run và **số shard khuyến nghị**.
2. **Chạy thật:** đặt `TIMING_ONLY = False`, `N_SHARDS` = số shard khuyến nghị, `SHARD = 0`, rồi *Save Version → Save & Run All*.
   Lặp lại với `SHARD = 1, 2, …`; mỗi shard là một version riêng.
3. Mỗi version cho ra `runs_shard{SHARD}.tar` (vài MB). Tải các file này từ tab *Output* của từng version và gửi cho Claude.
   Nếu một shard chưa chạy hết vì vượt ngân sách thời gian, chạy lại **đúng shard đó** và thêm `runs_shard{SHARD}.tar` cũ làm input: các run đã xong sẽ được bỏ qua.

In [ ]:
SHARD = 0            # 0 .. N_SHARDS-1
N_SHARDS = 3         # đặt theo kết quả đo thời gian
TIME_BUDGET_H = 8.0  # không bắt đầu job mới nếu nhiều khả năng vượt mốc này (an toàn với giới hạn session của Kaggle)
TIMING_ONLY = True   # True = chỉ chạy đo thời gian 3 epoch

In [ ]:
import os, glob, subprocess, json, math
subprocess.run('rm -rf /tmp/icme && git clone -q --depth 1 -b claude/eager-noether-4f8c81 https://github.com/ptrnghieu/icme.git /tmp/icme', shell=True, check=True)
os.chdir('/tmp/icme/pilot')
print(subprocess.run('git -C /tmp/icme log -1 --format=%h_%cd', shell=True, capture_output=True, text=True).stdout)
tars = glob.glob('/kaggle/input/**/cremad_processed.tar', recursive=True)
assert tars, 'Chưa thêm output của notebook 01 làm input'
os.makedirs('/tmp/cremad', exist_ok=True)
subprocess.run(f'tar -xf {tars[0]} -C /tmp/cremad', shell=True, check=True)
# khôi phục các run đã xong của shard này (nếu có gắn runs_shard*.tar cũ làm input)
os.makedirs('/tmp/runs', exist_ok=True)
for t in glob.glob('/kaggle/input/**/runs_shard*.tar', recursive=True):
    subprocess.run(f'tar -xf {t} -C /tmp', shell=True, check=True)
    print('restored', t)
NPROC = os.cpu_count()
WORKERS = max(1, NPROC // 2)
print('CPUs:', NPROC, '| dataloader workers per job:', WORKERS)
subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv', shell=True)

In [ ]:
if TIMING_ONLY:
    # 2 job chạy song song (mỗi GPU một job), giống điều kiện khi chạy thật
    jobs = [f'--root /tmp/cremad --out /tmp/runs_timing --split standard --method ogm_ge --alpha 0.5 --seed {s} --epochs 3 --amp --workers {WORKERS}' for s in (0, 1)]
    open('/tmp/jobs_timing.txt', 'w').write('\n'.join(jobs) + '\n')
    subprocess.run('python launch.py --jobs /tmp/jobs_timing.txt --gpus 0,1 --per_gpu 1', shell=True, check=True)
    secs = []
    for f in glob.glob('/tmp/runs_timing/*/log.jsonl'):
        recs = [json.loads(l) for l in open(f)]
        secs += [r['sec'] for r in recs[1:]]   # bỏ epoch đầu (khởi động)
    sec = sum(secs) / len(secs)
    h_run = 100 * sec / 3600 * 1.05            # 100 epoch, cộng 5% cho bước dump cuối
    shards = math.ceil(49 * h_run / (2 * TIME_BUDGET_H * 0.9))
    print(f'giây/epoch (2 job song song): {sec:.1f} | ước tính mỗi run: {h_run:.2f} h | '
          f'tổng 49 run trên 2 GPU: {49*h_run/2:.1f} h session | N_SHARDS khuyến nghị: {shards}')

In [ ]:
if not TIMING_ONLY:
    subprocess.run(f'python make_jobs.py --root /tmp/cremad --out /tmp/runs --jobs /tmp/jobs.txt --extra "--amp --workers {WORKERS}"', shell=True, check=True)
    subprocess.run(f'python launch.py --jobs /tmp/jobs.txt --gpus 0,1 --per_gpu 1 --shard {SHARD} --n_shards {N_SHARDS} --time_budget_h {TIME_BUDGET_H}', shell=True, check=True)
    subprocess.run(f"tar -cf /kaggle/working/runs_shard{SHARD}.tar -C /tmp --exclude='*.pt' runs", shell=True, check=True)
    subprocess.run('python analyze.py --root /tmp/cremad --runs /tmp/runs --out /kaggle/working/pilot_report_partial.md', shell=True)
    subprocess.run('ls -la /kaggle/working', shell=True)